In [38]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from yfinance import download
from scipy.stats import chi2

In [ ]:
param = {
    'ticker_ativo': 'KLBN4.SA',
    'ticker_indice': '^BVSP',
    'interval': '1mo',
    'period': 'max',
    'column': 'Close'
}

tickers = [param['ticker_indice'], param['ticker_ativo']]

df = download(tickers, interval=param['interval'], period=param['period'], progress=False, auto_adjust=False)[param['column']]
df = df.dropna()

for ticker in tickers:
    df[f'{ticker}_norm'] = df[f'{ticker}'] / df[f'{ticker}'].iloc[0]
    df[f'{ticker}_ret'] = df[f'{ticker}'].pct_change()

df['corr_price'] = df[param['ticker_ativo']].rolling(21).corr(df[param['ticker_indice']])
df['corr_ret'] = df[f"{param['ticker_ativo']}_ret"].rolling(21).corr(df[f"{param['ticker_indice']}_ret"])
    


In [30]:
df.tail(3)

Ticker,KLBN4.SA,^BVSP,^BVSP_norm,^BVSP_ret,KLBN4.SA_norm,KLBN4.SA_ret,corr_price,corr_ret
Date,,,,,,,,
2025-12-01,3.76,161125.000000,2.708438,0.012906,3.429690,0.066742,-0.684919,-0.026023
2026-01-01,3.85,181364.000000,3.048647,0.125611,3.511784,0.023936,-0.542917,-0.009452
2026-02-01,4.05,186464.296875,3.134381,0.028122,3.694214,0.051948,-0.396475,-0.038614


In [31]:
# criar subplots
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    row_heights=[0.7, 0.3],
    subplot_titles=("Performance Normalizada", "Correlação 21 períodos")
)

# =====================
# gráfico superior
# =====================

fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df[f"{param['ticker_ativo']}_norm"],
        name=param['ticker_ativo'],
        line=dict(color="blue")
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df[f"{param['ticker_indice']}_norm"],
        name=param['ticker_indice'],
        line=dict(color="black")
    ),
    row=1, col=1
)


# =====================
# gráfico inferior
# =====================

fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df['corr_price'],
        name="Correlação Preço",
        line=dict(color="green")
    ),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df['corr_ret'],
        name="Correlação Retorno",
        line=dict(color="red")
    ),
    row=2, col=1
)

# linha zero correlação
fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="gray",
    row=2,
    col=1

)

# layout
fig.update_layout(
    template="plotly_white",
    height=700,
    title=f"{param['ticker_ativo']} vs {param['ticker_indice']}",
    legend=dict(
        orientation="h",
        y=-0.15,
        x=0.5,
        xanchor="center"
    )
)

fig.show()


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import chi2


# =====================
# DADOS
# =====================

x = df[f'{param["ticker_ativo"]}_ret'].dropna()
y = df.loc[x.index, f'{param["ticker_indice"]}_ret']

ret_ativo_atual = x.iloc[-1]
ret_indice_atual = y.iloc[-1]


# =====================
# REGRESSÃO LINEAR
# =====================

coef = np.polyfit(x.values, y.values, 1)
poly = np.poly1d(coef)

x_reg = np.linspace(x.min(), x.max(), 200)
y_reg = poly(x_reg)


# =====================
# ELIPSE 95%
# =====================

cov = np.cov(x.values, y.values)

vals, vecs = np.linalg.eig(cov)

order = vals.argsort()[::-1]

vals = vals[order]
vecs = vecs[:, order]

theta = np.linspace(0, 2 * np.pi, 300)

chi_square_val = np.sqrt(chi2.ppf(0.95, 2))

ellipse = vecs @ np.vstack([
    chi_square_val * np.sqrt(vals[0]) * np.cos(theta),
    chi_square_val * np.sqrt(vals[1]) * np.sin(theta)
])

ellipse_x_rot = ellipse[0] + x.mean()
ellipse_y_rot = ellipse[1] + y.mean()


# =====================
# FIGURA BASE (Plotly Express)
# =====================

fig = px.scatter(
    x=x,
    y=y,
    template="plotly_white",
    height=650
)

fig.update_traces(
    name="Observações",
    marker=dict(
        size=7,
        color="#28527A",
        opacity=0.55,
        line=dict(color="white", width=0.7)
    ),
    showlegend=True
)


# =====================
# REGRESSÃO
# =====================

fig.add_trace(
    go.Scatter(
        x=x_reg,
        y=y_reg,
        mode="lines",
        name="Regressão Linear",
        line=dict(
            color="#444444",
            dash="dash",
            width=2
        )
    )
)


# =====================
# ELIPSE
# =====================

fig.add_trace(
    go.Scatter(
        x=ellipse_x_rot,
        y=ellipse_y_rot,
        mode="lines",
        name="Elipse 95%",
        line=dict(
            color="#7B2D26",
            dash="dash",
            width=2
        )
    )
)


# =====================
# PONTO ATUAL
# =====================

fig.add_trace(
    go.Scatter(
        x=[ret_ativo_atual],
        y=[ret_indice_atual],
        mode="markers",
        name="Ponto Atual",
        marker=dict(
            size=14,
            color="#B4161B",
            line=dict(color="black", width=1.2)
        )
    )
)


# =====================
# ANOTAÇÃO
# =====================

fig.add_annotation(
    x=ret_ativo_atual,
    y=ret_indice_atual,
    text=f"({ret_ativo_atual:.4f}, {ret_indice_atual:.4f})",
    showarrow=True,
    arrowhead=2,
    ax=20,
    ay=-30,
    font=dict(size=12, color="#B4161B"),
    bgcolor="white",
    bordercolor="#B4161B",
    borderwidth=1
)


# =====================
# LAYOUT FINAL
# =====================

fig.update_layout(
    title=dict(
        text=f"Relação entre Retornos {param['ticker_ativo']} vs {param['ticker_indice']}",
        x=0.5
    ),
    xaxis_title=f"{param['ticker_ativo']} Retorno",
    yaxis_title=f"{param['ticker_indice']} Retorno",
    legend_title=None,
    template="plotly_white",
    height=650
)

fig.update_xaxes(showgrid=True)
fig.update_yaxes(showgrid=True)

fig.show()
